# Course 1, Module 2: Risk Buffers
## Where Should DDC Deploy Fogging Teams?

**Scenario:** DDC has confirmed 30 dengue cases across three provinces. Mosquito fogging teams need deployment zones. Budget is limited.

**Key Questions:**
1. Where are the risk rings around each case?
2. Which schools and hospitals fall inside those rings?
3. What is the total area that needs fogging?
4. How do we prioritize by severity tier?

In [ ]:
from ddc_helpers import run_query, show_on_map, show_buffers

---

## Part 1: Risk Buffer Concept

### How Buffers Work

DDC uses **3 risk tiers** for vector control:
- **RED** (0-500m): Immediate fogging, larval survey
- **ORANGE** (500m-1km): Fogging within 48 hours
- **YELLOW** (1-2km): Enhanced surveillance, community awareness

### Important: GEOMETRY vs. GEOMETRY

`ST_Buffer()` only works with **GEOMETRY** (flat coordinates), not **GEOMETRY** (lat/lon on Earth's surface).

Since our data is stored as `GEOMETRY`, we convert using:
```sql
ST_GeomFromText(ST_AsText(geom))
```

Buffer distances are in **degrees**. At Thai latitudes (~14°N):
- **1 degree ≈ 111,000 meters**

For precise distance checks (e.g., "schools within 500m"), we still use `ST_Distance()` on GEOMETRY.

In [ ]:
# Let's see what a single 500m buffer looks like
run_query("""
SELECT case_id, severity,
    ST_AsText(ST_Buffer(ST_GeomFromText(ST_AsText(geom)), 500.0 / 111000)) AS buffer_500m_wkt
FROM ddc_training.dengue_cases
WHERE case_id = 1
""")

**Explanation:** That WKT polygon is a circle with ~500m radius around case #1. In a GIS viewer, this would appear as a red ring around the patient's location.

---

## Part 2: Schools in Danger Zone

DDC sends inspectors to schools inside risk buffers to check for *Aedes aegypti* breeding containers (water jars, tires, flower pots).

We use `ST_Distance()` on GEOMETRY for accurate meter-based distances.

In [ ]:
# Which schools are near dengue cases?
run_query("""
SELECT
    s.name AS school_name, s.district,
    d.case_id, d.severity, d.infection_date,
    ROUND(ST_Distance(s.geom, d.geom) * 111000, 0) AS distance_meters,
    CASE
        WHEN ST_Distance(s.geom, d.geom) <= 500.0 / 111000 THEN 'RED (0-500m)'
        WHEN ST_Distance(s.geom, d.geom) <= 1000.0 / 111000 THEN 'ORANGE (500m-1km)'
        WHEN ST_Distance(s.geom, d.geom) <= 2000.0 / 111000 THEN 'YELLOW (1-2km)'
    END AS risk_tier
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geom, d.geom) <= 2000.0 / 111000
ORDER BY s.name, distance_meters
""")

**Insight:** Schools that appear multiple times are near **multiple cases**. These are highest priority for DDC inspection teams.

In [ ]:
# Aggregate: How many cases threaten each school?
run_query("""
SELECT
    s.name AS school_name, s.district,
    COUNT(*) AS threatening_cases,
    SUM(CASE WHEN d.severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_nearby,
    ROUND(MIN(ST_Distance(s.geom, d.geom)) * 111000, 0) AS nearest_case_meters
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geom, d.geom) <= 2000.0 / 111000
GROUP BY s.name, s.district
ORDER BY threatening_cases DESC
""")

---

## Part 3: Visualize Risk Buffers on Map

In [ ]:
# Show all risk buffers on an interactive map
show_buffers("""
SELECT
    province_name, risk_tier, radius_meters, case_count,
    area_sq_km, ST_AsText(zone_geom) AS wkt
FROM ddc_training.dengue_risk_zones
ORDER BY radius_meters
""")

---

## Part 4: Area Calculation

### Converting Square Degrees to Square Kilometers

`ST_Area()` on GEOMETRY returns results in **square degrees**.

Conversion formula at Thai latitude (~14°N):
- 1 degree latitude ≈ 111 km
- 1 degree longitude ≈ 108 km (cos(14°) × 111 km)
- **1 square degree ≈ 111 × 108 ≈ 11,988 square km**

**Multiply square degrees by 11,988 to get square kilometers.**

In [ ]:
# Total area needing treatment by province and risk tier
run_query("""
SELECT
    province_name, risk_tier, case_count,
    area_sq_km
FROM ddc_training.dengue_risk_zones
ORDER BY province_name, radius_meters
""")

---

## Part 5: Overlap Savings

### Why Merge Buffers?

**Problem:** When cases are clustered, their 500m buffers overlap. Fogging teams don't need to spray the same area twice.

**Solution:** `ST_Union()` merges overlapping polygons into a single shape.

**Benefit:** Compare the "naive sum" (all circles, with overlap) vs. the "merged area" (overlap removed). The difference is **overlap savings**—resources saved by smart fogging deployment.

In [ ]:
# Calculate overlap savings per province
run_query("""
SELECT
    pb.province_name,
    ROUND(SUM(ST_Area(ST_Buffer(d.geom, 500.0/111000)))*11988, 3) AS naive_sum_sq_km,
    ROUND(ST_Area(ST_Union(ST_Buffer(d.geom, 500.0/111000)))*11988, 3) AS merged_sq_km,
    ROUND((1 - ST_Area(ST_Union(ST_Buffer(d.geom, 500.0/111000)))
         / NULLIFZERO(SUM(ST_Area(ST_Buffer(d.geom, 500.0/111000))))) * 100, 1) AS overlap_savings_pct
FROM ddc_training.dengue_cases d
JOIN ddc_training.province_boundaries pb
    ON ST_Intersects(pb.boundary, d.geom)
GROUP BY pb.province_name
ORDER BY overlap_savings_pct DESC
""")

---

## Key Functions Summary

| Function | What it does |
|---|---|
| `ST_Buffer(geom, degrees)` | Creates a circle polygon at given radius |
| `ST_Intersects(a, b)` | TRUE if two spatial objects overlap |
| `ST_Union(geom)` | Merges multiple polygons, removes overlap |
| `ST_Area(geom)` | Area in square degrees (GEOMETRY) |
| `ST_Distance(geom, geom)` | Proximity check in meters (GEOMETRY) |
| `ST_GeomFromText(ST_AsText(geom))` | Convert GEOMETRY to GEOMETRY for buffering |
| `ST_AsText(geom)` | Convert geometry to WKT text format |

---

## Exercise: Fogging Priority Ranking

### Your Task

Create a **priority ranking for fogging deployment** by calculating a "risk score" per school:

**Scoring Rules:**
- **3 points** for each RED-tier case nearby (< 500m)
- **2 points** for each ORANGE-tier case (500m - 1km)
- **1 point** for each YELLOW-tier case (1km - 2km)
- **Double the points** if case severity is DHF or DSS (severe dengue)

**Output:** Rank schools by risk score (highest first).

**Questions to Answer:**
1. Which 3 schools get fogging first?
2. What is the average risk score?
3. How many schools are in the RED zone?

**Hints:**
- Use nested CASE statements to assign points based on distance tiers
- Use `SUM()` with `CASE` to double severe cases (DHF/DSS)
- Join `schools`, `dengue_cases`, and use `ST_Distance()` for distance calculations
- GROUP BY school, then ORDER BY risk score DESC

In [ ]:
# Write your SQL query below
# Use run_query() to execute it

run_query("""
-- Your SQL here

""")

---

### Model Answer

<details>
<summary><b>Click to reveal the answer</b></summary>

```sql
SELECT
    s.name AS school_name,
    s.district,
    COUNT(*) AS case_count,
    SUM(
        CASE
            WHEN ST_Distance(s.geom, d.geom) <= 500.0 / 111000 THEN 3
            WHEN ST_Distance(s.geom, d.geom) <= 1000.0 / 111000 THEN 2
            WHEN ST_Distance(s.geom, d.geom) <= 2000.0 / 111000 THEN 1
        END *
        CASE
            WHEN d.severity IN ('DHF', 'DSS') THEN 2
            ELSE 1
        END
    ) AS risk_score
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d ON ST_Distance(s.geom, d.geom) <= 2000.0 / 111000
GROUP BY s.name, s.district
ORDER BY risk_score DESC, case_count DESC
```

**Expected Results:**
- Top schools by risk score represent highest-priority fogging zones
- Schools with severe cases (DHF/DSS) nearby score higher
- Multiple threats in RED zone amplify priority

</details>